In [1]:
import sys
sys.path.append('../backend')

from data.cf_api import get_user_info, get_user_submissions, get_user_rating_history
from data.data_processor import process_user_profile

handle  = "Harsh_18"    # change to your handle for personal insight
info    = get_user_info(handle)
subs    = get_user_submissions(handle)
history = get_user_rating_history(handle)
profile = process_user_profile(info, subs, history)

print(f"Tags analyzed: {len(profile['tag_features'])}")
print(f"All tags: {list(profile['tag_features'].keys())}")

Tags analyzed: 31
All tags: ['implementation', 'greedy', 'math', 'brute force', 'dp', '*special', 'strings', 'sortings', 'constructive algorithms', 'graph matchings', 'shortest paths', 'dfs and similar', 'graphs', 'binary search', 'two pointers', 'bitmasks', 'geometry', 'data structures', 'number theory', 'expression parsing', 'games', 'hashing', 'probabilities', 'ternary search', 'combinatorics', 'schedules', 'dsu', 'trees', '2-sat', 'fft', 'interactive']


In [2]:
import pandas as pd

rows = []
for tag, f in profile['tag_features'].items():
    rows.append({
        'tag':               tag,
        'solved':            f['total_solved'],
        'solve_rate':        f['solve_rate'],
        'avg_attempts':      f['avg_attempts'],
        'highest_rating':    f['highest_rating'],
        'avg_rating':        f['avg_rating_solved'],
        'recency_days':      f['recency_days'],
        'contest_%':         round(f['contest_solve_percentage']*100, 1),
        'first_try_%':       round(f['first_try_ac_rate']*100, 1),
        'recent_avg':        f['recent_avg_rating'],
        'momentum':          f['momentum']
    })

df = pd.DataFrame(rows).sort_values('solve_rate', ascending=False)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
print(df.to_string())

                        tag  solved  solve_rate  avg_attempts  highest_rating  avg_rating  recency_days  contest_%  first_try_%  recent_avg  momentum
5                  *special       3        1.00          4.67            1200     1033.30        725.00       0.00         0.00     1033.30      0.00
9           graph matchings       3        1.00          1.67            1300     1100.00        538.90       0.00        66.70     1100.00      0.00
13            binary search      39        1.00          1.97            1700     1197.40         12.90      23.10        48.70     1380.00    182.60
11          dfs and similar       8        1.00          1.75            1600     1250.00         17.90      37.50        50.00     1280.00     30.00
12                   graphs       5        1.00          1.80            1400     1180.00        163.90      40.00        40.00     1180.00      0.00
10           shortest paths       2        1.00          1.00            1400     1100.00        843

In [4]:
print("=== AVG RATING SOLVED ===")
print(df['avg_rating'].describe())
print()

print("=== CONTEST SOLVE % ===")
print(df['contest_%'].describe())
print()

print("=== FIRST TRY % ===")
print(df['first_try_%'].describe())
print()

print("=== MOMENTUM ===")
print(df['momentum'].describe())
print()

# Find tags with positive momentum (improving)
improving = df[df['momentum'] > 100].sort_values('momentum', ascending=False)
print("IMPROVING TOPICS (momentum > 100):")
print(improving[['tag', 'avg_rating', 'recent_avg', 'momentum']])
print()

# Find tags with negative momentum (declining)
declining = df[df['momentum'] < -40].sort_values('momentum')
print("DECLINING TOPICS (momentum < -100):")
print(declining[['tag', 'avg_rating', 'recent_avg', 'momentum']])

=== AVG RATING SOLVED ===
count     31.00
mean    1158.45
std      210.90
min      800.00
25%     1027.55
50%     1106.70
75%     1210.00
max     1700.00
Name: avg_rating, dtype: float64

=== CONTEST SOLVE % ===
count    31.00
mean     15.40
std      19.72
min       0.00
25%       0.00
50%      12.50
75%      22.90
max     100.00
Name: contest_%, dtype: float64

=== FIRST TRY % ===
count    31.00
mean     58.97
std      28.44
min       0.00
25%      46.45
50%      52.80
75%      76.40
max     100.00
Name: first_try_%, dtype: float64

=== MOMENTUM ===
count     31.00
mean      24.56
std       81.92
min     -116.60
25%        0.00
50%        0.00
75%       35.20
max      292.60
Name: momentum, dtype: float64

IMPROVING TOPICS (momentum > 100):
               tag  avg_rating  recent_avg  momentum
0   implementation      927.40     1220.00    292.60
13   binary search     1197.40     1380.00    182.60
14    two pointers     1095.50     1260.00    164.50
4               dp     1167.50     1

In [5]:
import numpy as np

MAX_RATING = 3500
VOLUME_CAP = 50

def compute_skill_score_v2(row):
    # Normalize all features to 0-1
    solve_score          = row['solve_rate']
    avg_rating_score     = min(row['avg_rating'] / MAX_RATING, 1.0)
    highest_rating_score = min(row['highest_rating'] / MAX_RATING, 1.0)
    recency_score        = np.exp(-row['recency_days'] / 30)
    volume_score         = min(row['solved'] / VOLUME_CAP, 1.0)
    first_try_score      = row['first_try_%'] / 100
    attempt_score        = 1 / (1 + row['avg_attempts'])

    # New 7-weight formula
    score = (
        0.20 * solve_score          +
        0.20 * avg_rating_score     +
        0.15 * highest_rating_score +
        0.15 * recency_score        +
        0.10 * volume_score         +
        0.10 * first_try_score      +
        0.10 * attempt_score
    )
    return round(score, 4)

df['skill_score'] = df.apply(compute_skill_score_v2, axis=1)
df = df.sort_values('skill_score')

print("=== WEAKEST TOPICS ===")
print(df.head(5)[['tag', 'skill_score', 'solve_rate', 'avg_rating', 'recency_days']])
print()
print("=== STRONGEST TOPICS ===")
print(df.tail(5)[['tag', 'skill_score', 'solve_rate', 'avg_rating', 'recency_days']])

=== WEAKEST TOPICS ===
              tag  skill_score  solve_rate  avg_rating  recency_days
5        *special         0.33        1.00     1033.30        725.00
25      schedules         0.36        1.00     1400.00        632.80
22  probabilities         0.40        1.00     1220.00        417.40
21        hashing         0.40        1.00     1350.00        632.80
12         graphs         0.41        1.00     1180.00        163.90

=== STRONGEST TOPICS ===
                        tag  skill_score  solve_rate  avg_rating  recency_days
18            number theory         0.60        1.00     1056.60         17.90
3               brute force         0.60        0.99      993.90         12.90
1                    greedy         0.60        0.97     1016.20         12.90
8   constructive algorithms         0.62        0.98     1030.10         12.90
2                      math         0.62        0.99      985.10         12.90


In [6]:
def old_score(row):
    solve_score    = row['solve_rate']
    attempt_score  = 1 / (1 + row['avg_attempts'])
    rating_score   = min(row['highest_rating'] / MAX_RATING, 1.0)
    recency_score  = np.exp(-row['recency_days'] / 30)
    return round(0.35*solve_score + 0.30*rating_score + 0.20*recency_score + 0.15*attempt_score, 4)

df['old_score'] = df.apply(old_score, axis=1)
df['score_diff'] = df['skill_score'] - df['old_score']

print("Tags where new score is SIGNIFICANTLY HIGHER than old (volume + first-try bonus):")
print(df[df['score_diff'] > 0.05][['tag', 'old_score', 'skill_score', 'score_diff', 'solved', 'first_try_%']])
print()
print("Tags where new score is SIGNIFICANTLY LOWER (avg rating matters now):")
print(df[df['score_diff'] < -0.05][['tag', 'old_score', 'skill_score', 'score_diff', 'avg_rating', 'highest_rating']])

Tags where new score is SIGNIFICANTLY HIGHER than old (volume + first-try bonus):
Empty DataFrame
Columns: [tag, old_score, skill_score, score_diff, solved, first_try_%]
Index: []

Tags where new score is SIGNIFICANTLY LOWER (avg rating matters now):
                   tag  old_score  skill_score  score_diff  avg_rating  \
5             *special       0.48         0.33       -0.15     1033.30   
25           schedules       0.49         0.36       -0.14     1400.00   
22       probabilities       0.53         0.40       -0.13     1220.00   
21             hashing       0.51         0.40       -0.12     1350.00   
12              graphs       0.52         0.41       -0.11     1180.00   
9      graph matchings       0.52         0.43       -0.09     1100.00   
19  expression parsing       0.49         0.43       -0.06      800.00   
23      ternary search       0.52         0.45       -0.07     1250.00   
16            geometry       0.54         0.46       -0.08     1106.70   
27       

In [7]:
print("=== MOMENTUM INSIGHT ===")
print("Tags where user is actively BREAKING THROUGH their ceiling:")
breakthrough = df[(df['momentum'] > 200) & (df['solve_rate'] > 0.4)]
print(breakthrough[['tag', 'avg_rating', 'recent_avg', 'momentum', 'highest_rating']])
print()
print("Interpretation:")
for _, row in breakthrough.iterrows():
    print(f"  {row['tag']}: all-time avg {row['avg_rating']:.0f} but last 5 problems avg {row['recent_avg']:.0f} → push harder!")

=== MOMENTUM INSIGHT ===
Tags where user is actively BREAKING THROUGH their ceiling:
              tag  avg_rating  recent_avg  momentum  highest_rating
0  implementation      927.40     1220.00    292.60            1600

Interpretation:
  implementation: all-time avg 927 but last 5 problems avg 1220 → push harder!
